# Inside the GPU: architecture and scheduling (chapter 4)

Session 12 (`12_gpu_architecture_scheduling`): session 11 launched grids of blocks of threads without asking where they run. This folder opens the hardware. It asks the GPU what it has, sizes grids against its limits, makes the threads of a block wait for each other, finds out which warp and lane each thread is, counts how branches split warps, and computes occupancy from the device properties.

This notebook goes **step by step**: 9 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/02_gpu_architecture/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 9: `01_device_query.cu`

The device query of the slides: `cudaGetDeviceCount` and `cudaGetDeviceProperties`, then the SM count, the block and grid limits and the per-SM limits (threads, blocks, registers, shared memory). A one-thread kernel reads the built-in `warpSize`, which must match the property. (slide: *Asking the GPU what it has*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_device_query.cu
#include "cuda_compat.h"

#include <cstdio>

__global__ void readWarpSize(int* deviceWarpSize) {
  *deviceWarpSize = warpSize;
}

static void printProperties(int device, const cudaDeviceProp& p) {
  std::printf("%d: %s, compute capability %d.%d\n", device, p.name, p.major, p.minor);
  std::printf("   %d SMs, %d threads/block, warp %d\n",
              p.multiProcessorCount,
              p.maxThreadsPerBlock,
              p.warpSize);
  std::printf("   block dimensions up to %d x %d x %d\n",
              p.maxThreadsDim[0],
              p.maxThreadsDim[1],
              p.maxThreadsDim[2]);
  std::printf("   grid dimensions up to %d x %d x %d\n",
              p.maxGridSize[0],
              p.maxGridSize[1],
              p.maxGridSize[2]);
  std::printf("   per SM: %d threads, %d blocks, %d registers, %zu KiB shared memory\n",
              p.maxThreadsPerMultiProcessor,
              p.maxBlocksPerMultiProcessor,
              p.regsPerMultiprocessor,
              p.sharedMemPerMultiprocessor / 1024);
  std::printf("   per block: %d registers, %zu KiB shared memory\n",
              p.regsPerBlock,
              p.sharedMemPerBlock / 1024);
}

static int warpSizeSeenByAKernel() {
  int warp = 0;
  int* warp_d;
  CHECK(cudaMalloc(&warp_d, sizeof(int)));

  launch(readWarpSize, dim3(1), dim3(1), warp_d);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(&warp, warp_d, sizeof(int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(warp_d));
  return warp;
}

static bool limitsMakeSense(const cudaDeviceProp& p) {
  const bool blockIsWholeWarps = p.maxThreadsPerBlock % p.warpSize == 0;
  const bool smHoldsABlock = p.maxThreadsPerMultiProcessor >= p.maxThreadsPerBlock;
  return p.multiProcessorCount > 0 && blockIsWholeWarps && smHoldsABlock;
}

int main() {
  int count = 0;
  CHECK(cudaGetDeviceCount(&count));
  std::printf("%d device(s)\n", count);

  bool ok = count > 0;
  for (int device = 0; device < count; ++device) {
    cudaDeviceProp p;
    CHECK(cudaGetDeviceProperties(&p, device));
    printProperties(device, p);
    ok = ok && limitsMakeSense(p);
  }

  if (count > 0) {
    cudaDeviceProp p;
    CHECK(cudaGetDeviceProperties(&p, 0));
    const int warp = warpSizeSeenByAKernel();
    std::printf("warpSize inside a kernel: %d, property warpSize: %d\n", warp, p.warpSize);
    ok = ok && warp == p.warpSize;
  }

  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_device_query.cu -o 01_device_query && ./01_device_query

**What to look at in the output:** On Colab's T4 you should see compute capability 7.5, 40 SMs, 1024 threads per block, and per SM 1024 threads, 16 blocks, 65536 registers and 64 KiB of shared memory (the CPU emulation reports fixed values of its own). The last line checks that the `warpSize` a kernel reads is the same 32 that the host property reports.

## Step 2 of 9: `02_blocks_needed.cu`

How many blocks a grid needs: `(n + threadsPerBlock - 1) / threadsPerBlock` for four launches, the idle threads of the last block, and which block owns each element. A block of 2048 threads is refused because the device allows `maxThreadsPerBlock` = 1024.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_blocks_needed.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

struct Launch {
  int n;
  int threadsPerBlock;
};

const Launch LAUNCHES[] = {{10, 4}, {16, 4}, {5, 8}, {8, 2048}};

__global__ void ownerKernel(int* ownerBlock, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    ownerBlock[i] = blockIdx.x;
  }
}

static int blocksToCover(int n, int threadsPerBlock) {
  return (n + threadsPerBlock - 1) / threadsPerBlock;
}

static std::vector<int> ownersOnDevice(int n, int threadsPerBlock) {
  const size_t bytes = n * sizeof(int);
  std::vector<int> ownerBlock(n, -1);

  int* ownerBlock_d;
  CHECK(cudaMalloc(&ownerBlock_d, bytes));

  launch(
      ownerKernel, dim3(blocksToCover(n, threadsPerBlock)), dim3(threadsPerBlock), ownerBlock_d, n);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(ownerBlock.data(), ownerBlock_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(ownerBlock_d));
  return ownerBlock;
}

static bool ownersAreCorrect(const std::vector<int>& ownerBlock, int threadsPerBlock) {
  for (int i = 0; i < (int)ownerBlock.size(); ++i) {
    if (ownerBlock[i] != i / threadsPerBlock) {
      return false;
    }
  }
  return true;
}

static bool runLaunch(const Launch& config, const cudaDeviceProp& p) {
  const int blocks = blocksToCover(config.n, config.threadsPerBlock);
  const int launched = blocks * config.threadsPerBlock;
  std::printf("n = %2d, %4d threads/block: %d blocks, %d threads, %d idle\n",
              config.n,
              config.threadsPerBlock,
              blocks,
              launched,
              launched - config.n);

  if (config.threadsPerBlock > p.maxThreadsPerBlock) {
    std::printf("   not launched: this device allows at most %d threads per block\n",
                p.maxThreadsPerBlock);
    return true;
  }

  const std::vector<int> ownerBlock = ownersOnDevice(config.n, config.threadsPerBlock);
  std::printf("   block of each element:");
  for (int owner : ownerBlock) {
    std::printf(" %d", owner);
  }
  std::printf("\n");
  return ownersAreCorrect(ownerBlock, config.threadsPerBlock);
}

int main() {
  cudaDeviceProp p;
  CHECK(cudaGetDeviceProperties(&p, 0));
  std::printf("blocks needed = (n + threadsPerBlock - 1) / threadsPerBlock\n\n");

  bool ok = true;
  for (const Launch& config : LAUNCHES) {
    ok = runLaunch(config, p) && ok;
  }

  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_blocks_needed.cu -o 02_blocks_needed && ./02_blocks_needed

**What to look at in the output:** For n = 10 with 4 threads per block, the owner list 0 0 0 0 1 1 1 1 2 2 shows that block 2 has only 2 of its 4 threads working; n = 16 is a multiple of 4 and leaves none idle. The 2048-thread launch is printed but refused because it exceeds `maxThreadsPerBlock` = 1024. Try: add `{20, 8}` to `LAUNCHES`: 3 blocks, 24 threads, 4 idle.

## Step 3 of 9: `03_sync_reverse.cu`

`reverseChunks`: each block copies its chunk of 8 elements to a `__shared__` array, waits at `__syncthreads()`, and reads another thread's slot. The barrier is outside any branch, and blocks never wait for each other. (slide: *Code: a barrier in a real kernel*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_sync_reverse.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

#define BLOCK 8

const int CHUNKS = 2;
const int N = CHUNKS * BLOCK;

__global__ void reverseChunks(int* d) {
  __shared__ int s[BLOCK];
  int t = threadIdx.x;
  int base = blockIdx.x * BLOCK;

  s[t] = d[base + t];
  __syncthreads();
  d[base + t] = s[BLOCK - 1 - t];
}

static void reverseOnDevice(std::vector<int>& data) {
  const size_t bytes = data.size() * sizeof(int);

  int* d_d;
  CHECK(cudaMalloc(&d_d, bytes));
  CHECK(cudaMemcpy(d_d, data.data(), bytes, cudaMemcpyHostToDevice));

  launch(reverseChunks, dim3(N / BLOCK), dim3(BLOCK), d_d);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(data.data(), d_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(d_d));
}

static std::vector<int> reverseOnCpu(const std::vector<int>& data) {
  std::vector<int> reversed(data.size());
  for (int chunk = 0; chunk < CHUNKS; ++chunk) {
    const int base = chunk * BLOCK;
    for (int t = 0; t < BLOCK; ++t) {
      reversed[base + t] = data[base + BLOCK - 1 - t];
    }
  }
  return reversed;
}

static void printData(const char* label, const std::vector<int>& data) {
  std::printf("%-8s", label);
  for (int i = 0; i < N; ++i) {
    if (i > 0 && i % BLOCK == 0) {
      std::printf(" |");
    }
    std::printf(" %3d", data[i]);
  }
  std::printf("\n");
}

int main() {
  std::vector<int> data(N);
  for (int i = 0; i < N; ++i) {
    data[i] = 10 * i;
  }
  const std::vector<int> expected = reverseOnCpu(data);

  std::printf("%d blocks of %d threads, each block reverses its own chunk\n", N / BLOCK, BLOCK);
  printData("before:", data);
  reverseOnDevice(data);
  printData("after:", data);

  const bool ok = data == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_sync_reverse.cu -o 03_sync_reverse && ./03_sync_reverse

**What to look at in the output:** Each half is reversed on its own: 0 … 70 becomes 70 … 0 and 80 … 150 becomes 150 … 80, and no value moves from one chunk to the other, because each block only sees its own `__shared__` array. Thread 0 writes 70, a value loaded by thread 7, which is why the barrier must come before the read.

## Step 4 of 9: `04_warps_and_lanes.cu`

`whoAmI` on 16×16 blocks: the row-major thread index `t = threadIdx.y * blockDim.x + threadIdx.x`, the warp `t / warpSize` and the lane `t % warpSize`. Every two rows of a 16-wide block form one warp; the sample threads are those of the slide's table. (slide: *Code: which warp and which lane am I?*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_warps_and_lanes.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int BLOCKS = 2;
const int BLOCK_X = 16;
const int BLOCK_Y = 16;
const int THREADS_PER_BLOCK = BLOCK_X * BLOCK_Y;
const int TOTAL_THREADS = BLOCKS * THREADS_PER_BLOCK;

struct Sample {
  int x;
  int y;
};

const Sample SAMPLES[] = {{0, 0}, {15, 1}, {0, 2}, {5, 3}, {15, 15}};

__global__ void whoAmI(int* warpOf, int* laneOf) {
  int t = threadIdx.y * blockDim.x + threadIdx.x;
  int warp = t / warpSize;
  int lane = t % warpSize;

  int g = blockIdx.x * (blockDim.x * blockDim.y) + t;
  warpOf[g] = warp;
  laneOf[g] = lane;
}

static void runWhoAmI(std::vector<int>& warpOf, std::vector<int>& laneOf) {
  const size_t bytes = TOTAL_THREADS * sizeof(int);

  int* warp_d;
  int* lane_d;
  CHECK(cudaMalloc(&warp_d, bytes));
  CHECK(cudaMalloc(&lane_d, bytes));

  launch(whoAmI, dim3(BLOCKS), dim3(BLOCK_X, BLOCK_Y), warp_d, lane_d);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(warpOf.data(), warp_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaMemcpy(laneOf.data(), lane_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(warp_d));
  CHECK(cudaFree(lane_d));
}

static void printWarpMap(const std::vector<int>& warpOf) {
  std::printf("warp of each thread of block 0 (row = threadIdx.y, column = threadIdx.x):\n");
  for (int y = 0; y < BLOCK_Y; ++y) {
    std::printf("  y=%2d:", y);
    for (int x = 0; x < BLOCK_X; ++x) {
      std::printf(" %d", warpOf[y * BLOCK_X + x]);
    }
    std::printf("\n");
  }
}

static void printSamples(const std::vector<int>& warpOf, const std::vector<int>& laneOf) {
  std::printf("\n  (x, y)      t  warp  lane\n");
  for (const Sample& s : SAMPLES) {
    const int t = s.y * BLOCK_X + s.x;
    std::printf("  (%2d, %2d)  %3d  %4d  %4d\n", s.x, s.y, t, warpOf[t], laneOf[t]);
  }
}

static bool matchesCpu(const std::vector<int>& warpOf,
                       const std::vector<int>& laneOf,
                       int warpSizeOfDevice) {
  for (int g = 0; g < TOTAL_THREADS; ++g) {
    const int t = g % THREADS_PER_BLOCK;
    if (warpOf[g] != t / warpSizeOfDevice || laneOf[g] != t % warpSizeOfDevice) {
      return false;
    }
  }
  return true;
}

int main() {
  cudaDeviceProp p;
  CHECK(cudaGetDeviceProperties(&p, 0));

  std::vector<int> warpOf(TOTAL_THREADS, -1);
  std::vector<int> laneOf(TOTAL_THREADS, -1);
  runWhoAmI(warpOf, laneOf);

  std::printf("%d blocks of %d x %d threads, warps of %d threads\n\n",
              BLOCKS,
              BLOCK_X,
              BLOCK_Y,
              p.warpSize);
  printWarpMap(warpOf);
  printSamples(warpOf, laneOf);

  const bool ok = matchesCpu(warpOf, laneOf, p.warpSize);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_warps_and_lanes.cu -o 04_warps_and_lanes && ./04_warps_and_lanes

**What to look at in the output:** In the map every two rows share a warp number (rows 0 and 1 are warp 0, rows 14 and 15 are warp 7), because a 16-wide block packs 32 consecutive values of t into one warp. In the sample table, thread (15, 1) has t = 31, the last lane of warp 0, and (0, 2) has t = 32, lane 0 of warp 1.

## Step 5 of 9: `05_divergence.cu`

Control divergence counted: the `branches` kernel records which way each thread goes at the boundary check, at `t % 2 == 0` and at `(t / warpSize) % 2 == 0`, and the host counts the lanes of every warp on each side. A branch costs 2 passes in a warp only when both sides have lanes. (slides: *Code: divergent and warp-uniform branches*, *The most common divergence: the boundary check*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_divergence.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int N = 80;
const int BLOCK_SIZE = 64;
const int BLOCKS = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;
const int LAUNCHED = BLOCKS * BLOCK_SIZE;
const int WARP_SIZE = 32;
const int NOT_RUN = -1;
const int IF_PATH = 1;
const int ELSE_PATH = 0;

using Path = std::vector<int>;

__global__ void branches(int* returnsEarly, int* evenThread, int* evenWarp, int n) {
  int t = threadIdx.x;
  int i = blockIdx.x * blockDim.x + t;

  if (i >= n) {
    returnsEarly[i] = IF_PATH;
    return;
  }
  returnsEarly[i] = ELSE_PATH;

  if (t % 2 == 0) {
    evenThread[i] = IF_PATH;
  } else {
    evenThread[i] = ELSE_PATH;
  }

  if ((t / warpSize) % 2 == 0) {
    evenWarp[i] = IF_PATH;
  } else {
    evenWarp[i] = ELSE_PATH;
  }
}

static int* deviceCopy(const std::vector<int>& host) {
  int* device;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

static void copyBack(std::vector<int>& host, int* device) {
  CHECK(cudaMemcpy(host.data(), device, host.size() * sizeof(int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(device));
}

static int lanesWith(const Path& path, int warp, int value) {
  int lanes = 0;
  for (int i = warp * WARP_SIZE; i < (warp + 1) * WARP_SIZE; ++i) {
    lanes += path[i] == value ? 1 : 0;
  }
  return lanes;
}

static void printBranch(const Path& path, int warp) {
  const int taken = lanesWith(path, warp, IF_PATH);
  const int notTaken = lanesWith(path, warp, ELSE_PATH);
  const int passes = (taken > 0 ? 1 : 0) + (notTaken > 0 ? 1 : 0);
  std::printf("   %2d/%2d %d pass%s", taken, notTaken, passes, passes == 1 ? "  " : "es");
}

struct Run {
  Path returnsEarly = Path(LAUNCHED, NOT_RUN);
  Path evenThread = Path(LAUNCHED, NOT_RUN);
  Path evenWarp = Path(LAUNCHED, NOT_RUN);
};

static Run runBranches() {
  Run run;
  int* returnsEarly_d = deviceCopy(run.returnsEarly);
  int* evenThread_d = deviceCopy(run.evenThread);
  int* evenWarp_d = deviceCopy(run.evenWarp);

  launch(branches, dim3(BLOCKS), dim3(BLOCK_SIZE), returnsEarly_d, evenThread_d, evenWarp_d, N);
  CHECK(cudaGetLastError());

  copyBack(run.returnsEarly, returnsEarly_d);
  copyBack(run.evenThread, evenThread_d);
  copyBack(run.evenWarp, evenWarp_d);
  return run;
}

static void printWarps(const Run& run) {
  std::printf("warp  threads    i >= n            t %% 2 == 0         (t / warpSize) %% 2 == 0\n");
  for (int warp = 0; warp < LAUNCHED / WARP_SIZE; ++warp) {
    std::printf("%4d  %3d-%3d", warp, warp * WARP_SIZE, (warp + 1) * WARP_SIZE - 1);
    printBranch(run.returnsEarly, warp);
    printBranch(run.evenThread, warp);
    printBranch(run.evenWarp, warp);
    std::printf("\n");
  }
}

static int expectedPath(bool taken, bool runs) {
  if (!runs) {
    return NOT_RUN;
  }
  return taken ? IF_PATH : ELSE_PATH;
}

static bool matchesCpu(const Run& run) {
  for (int i = 0; i < LAUNCHED; ++i) {
    const int t = i % BLOCK_SIZE;
    const bool runs = i < N;

    if (run.returnsEarly[i] != expectedPath(!runs, true) ||
        run.evenThread[i] != expectedPath(t % 2 == 0, runs) ||
        run.evenWarp[i] != expectedPath((t / WARP_SIZE) % 2 == 0, runs)) {
      return false;
    }
  }
  return true;
}

int main() {
  const Run run = runBranches();

  std::printf("n = %d, %d blocks of %d threads\n", N, BLOCKS, BLOCK_SIZE);
  std::printf("each branch: lanes taking if / lanes taking else, passes the warp needs\n\n");
  printWarps(run);

  const bool ok = matchesCpu(run);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_divergence.cu -o 05_divergence && ./05_divergence

**What to look at in the output:** Each cell is "lanes taking the if / lanes taking the else". `t % 2 == 0` splits every running warp 16/16, so it always costs 2 passes; the warp-aligned test `(t / warpSize) % 2 == 0` never splits a warp. The boundary check splits only warp 2 (threads 64 to 79 work, 80 to 95 return), and warp 3 shows 0/0 in the later branches because all its lanes already returned. Try: set `N = 96`; warp 2 no longer diverges at `i >= n`.

## Step 6 of 9: `06_occupancy.cu`

An occupancy calculator: blocks per SM limited by threads, block slots and registers, and occupancy as resident warps over the maximum. It reproduces the slides' H100 table (the check), then runs the same table with the limits that `cudaGetDeviceProperties` reports for this device. (slides: *Occupancy in numbers: four ways to lose it*, *Code: let the runtime compute occupancy*; the runtime's `cudaOccupancy*` calls themselves need a real GPU)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 06_occupancy.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <string>

struct SmLimits {
  int threads;
  int blocks;
  int registers;
  int warpSize;
};

struct Kernel {
  int threadsPerBlock;
  int registersPerThread;
  int expectedBlocks;
  const char* expectedLimit;
};

struct Occupancy {
  int blocks;
  int percent;
  std::string limit;
};

const SmLimits H100 = {2048, 32, 65536, 32};
const int FEW_REGISTERS = 16;

const Kernel SLIDE_TABLE[] = {
    {1024, FEW_REGISTERS, 2, "threads"},
    {512, FEW_REGISTERS, 4, "threads"},
    {256, FEW_REGISTERS, 8, "threads"},
    {128, FEW_REGISTERS, 16, "threads"},
    {64, FEW_REGISTERS, 32, "threads and blocks"},
    {32, FEW_REGISTERS, 32, "blocks"},
    {768, FEW_REGISTERS, 2, "threads"},
    {256, 64, 4, "registers"},
    {512, 31, 4, "threads and registers"},
    {512, 33, 3, "registers"},
};

static void addLimit(std::string& limits, const char* name) {
  if (!limits.empty()) {
    limits += " and ";
  }
  limits += name;
}

static Occupancy occupancy(const SmLimits& sm, const Kernel& kernel) {
  const int byThreads = sm.threads / kernel.threadsPerBlock;
  const int byBlocks = sm.blocks;
  const int byRegisters = sm.registers / (kernel.registersPerThread * kernel.threadsPerBlock);

  Occupancy result;
  result.blocks = std::min(byThreads, std::min(byBlocks, byRegisters));

  const int warpsPerBlock = (kernel.threadsPerBlock + sm.warpSize - 1) / sm.warpSize;
  const int maxWarps = sm.threads / sm.warpSize;
  result.percent = 100 * result.blocks * warpsPerBlock / maxWarps;

  if (byThreads == result.blocks) {
    addLimit(result.limit, "threads");
  }
  if (byBlocks == result.blocks) {
    addLimit(result.limit, "blocks");
  }
  if (byRegisters == result.blocks) {
    addLimit(result.limit, "registers");
  }
  return result;
}

static bool printTable(const SmLimits& sm) {
  std::printf(
      "SM limits: %d threads, %d blocks, %d registers\n", sm.threads, sm.blocks, sm.registers);
  std::printf("threads/block  registers/thread  blocks/SM  occupancy  limited by\n");

  bool matchesSlide = true;
  for (const Kernel& kernel : SLIDE_TABLE) {
    const Occupancy result = occupancy(sm, kernel);
    std::printf("%13d  %16d  %9d  %8d%%  %s\n",
                kernel.threadsPerBlock,
                kernel.registersPerThread,
                result.blocks,
                result.percent,
                result.limit.c_str());
    matchesSlide = matchesSlide && result.blocks == kernel.expectedBlocks &&
                   result.limit == kernel.expectedLimit;
  }
  std::printf("\n");
  return matchesSlide;
}

int main() {
  std::printf("H100, as on the slides:\n");
  const bool ok = printTable(H100);

  cudaDeviceProp p;
  CHECK(cudaGetDeviceProperties(&p, 0));
  const SmLimits thisDevice = {p.maxThreadsPerMultiProcessor,
                               p.maxBlocksPerMultiProcessor,
                               p.regsPerMultiprocessor,
                               p.warpSize};
  std::printf("this device, %s:\n", p.name);
  printTable(thisDevice);

  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 06_occupancy.cu -o 06_occupancy && ./06_occupancy

**What to look at in the output:** In the H100 table, 32-thread blocks reach only 50% because the 32-block limit is hit before the thread limit, and 512 threads at 33 registers drop to 3 blocks (75%) while 31 registers still allow 4. On Colab's T4 the second table changes: the SM holds only 1024 threads and 16 blocks, so the thread limit binds first and the two register rows (256 at 64, 512 at 33) reach 100%.

## Step 7 of 9: `07_barrier_placement.cu`

The right version of the slide's `step` kernel: even threads double `x[t]`, odd threads add 1, then all 8 threads meet at one `__syncthreads()` after the `if`/`else` and compute `y[t] = x[t] - x[t ^ 1]` from their neighbour's new value. The wrong version, with one barrier inside each branch, is not run: its behaviour is undefined. (slide: *A barrier every thread must reach: the wrong and the right way*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 07_barrier_placement.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int THREADS = 8;

using Values = std::vector<float>;

__global__ void step(float* x, float* y) {
  int t = threadIdx.x;
  if (t % 2 == 0) {
    x[t] *= 2.0f;
  } else {
    x[t] += 1.0f;
  }
  __syncthreads();
  y[t] = x[t] - x[t ^ 1];
}

struct StepResult {
  Values x;
  Values y;
};

static StepResult stepOnDevice(const Values& x) {
  const size_t bytes = THREADS * sizeof(float);
  StepResult result{Values(THREADS), Values(THREADS)};

  float* x_d;
  float* y_d;
  CHECK(cudaMalloc(&x_d, bytes));
  CHECK(cudaMalloc(&y_d, bytes));
  CHECK(cudaMemcpy(x_d, x.data(), bytes, cudaMemcpyHostToDevice));

  launch(step, dim3(1), dim3(THREADS), x_d, y_d);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(result.x.data(), x_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaMemcpy(result.y.data(), y_d, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(x_d));
  CHECK(cudaFree(y_d));
  return result;
}

static StepResult stepOnCpu(const Values& input) {
  StepResult result{input, Values(THREADS)};
  for (int t = 0; t < THREADS; ++t) {
    result.x[t] = t % 2 == 0 ? input[t] * 2.0f : input[t] + 1.0f;
  }
  for (int t = 0; t < THREADS; ++t) {
    result.y[t] = result.x[t] - result.x[t ^ 1];
  }
  return result;
}

static void printValues(const char* label, const Values& values) {
  std::printf("%-30s", label);
  for (float v : values) {
    std::printf(" %5.1f", v);
  }
  std::printf("\n");
}

int main() {
  Values x(THREADS);
  for (int t = 0; t < THREADS; ++t) {
    x[t] = static_cast<float>(10 * (t + 1));
  }

  std::printf("1 block of %d threads; even threads double x[t], odd threads add 1\n", THREADS);
  std::printf("one __syncthreads() after the if/else, reached by all %d threads\n\n", THREADS);
  std::printf("%-30s", "thread t:");
  for (int t = 0; t < THREADS; ++t) {
    std::printf(" %5d", t);
  }
  std::printf("\n");
  printValues("x before:", x);

  const StepResult result = stepOnDevice(x);
  const StepResult expected = stepOnCpu(x);
  printValues("x after the branch:", result.x);
  printValues("y[t] = x[t] - x[t ^ 1]:", result.y);
  std::printf("\nthread 0 reads x[1] = %.1f, written by thread 1 before the barrier\n",
              result.x[1]);
  std::printf(
      "the wrong version (one barrier in each branch) is not run: its behaviour is undefined\n");

  const bool ok = result.x == expected.x && result.y == expected.y;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 07_barrier_placement.cu -o 07_barrier_placement && ./07_barrier_placement

**What to look at in the output:** Each pair (t, t ^ 1) gives opposite values in y: thread 0 gets 20 − 21 = −1 and thread 1 gets 21 − 20 = 1, because thread 0 reads `x[1]` = 21 only after thread 1 has written it. The result goes to a second array `y`: writing it back into `x` would let a thread overwrite `x[t]` while its neighbour is still reading it, a race that ThreadSanitizer reports.

## Step 8 of 9: `08_divergence_2d.cu`

The 2D boundary check `col < width && row < height` on 16 × 16 blocks, where each warp is two rows of a block. The kernel records which threads are inside the picture; the host counts, for every warp, whether its lanes are all inside, all outside or split, for the 76 × 62 picture of the slide and the 100 × 70 image of exercise 12.8. (slide: *Boundary checks in 2D: which warps diverge?*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 08_divergence_2d.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const int BLOCK_SIDE = 16;
const int THREADS_PER_BLOCK = BLOCK_SIDE * BLOCK_SIDE;
const int WARP_SIZE = 32;
const int WARPS_PER_BLOCK = THREADS_PER_BLOCK / WARP_SIZE;
const int OUTSIDE = 0;
const int INSIDE = 1;

using Flags = std::vector<int>;

struct Picture {
  int width;
  int height;
};

const Picture PICTURES[] = {{76, 62}, {100, 70}};

__global__ void boundaryCheck(int* inside, int width, int height) {
  int col = blockIdx.x * BLOCK_SIDE + threadIdx.x;
  int row = blockIdx.y * BLOCK_SIDE + threadIdx.y;
  int t = threadIdx.y * blockDim.x + threadIdx.x;
  int slot = (blockIdx.y * gridDim.x + blockIdx.x) * THREADS_PER_BLOCK + t;

  if (col < width && row < height) {
    inside[slot] = INSIDE;
  } else {
    inside[slot] = OUTSIDE;
  }
}

static int blocksToCover(int n) {
  return (n + BLOCK_SIDE - 1) / BLOCK_SIDE;
}

static Flags boundaryCheckOnDevice(const Picture& picture) {
  const dim3 grid(blocksToCover(picture.width), blocksToCover(picture.height));
  const int threads = grid.x * grid.y * THREADS_PER_BLOCK;
  Flags inside(threads, OUTSIDE);

  int* inside_d;
  CHECK(cudaMalloc(&inside_d, threads * sizeof(int)));
  launch(
      boundaryCheck, grid, dim3(BLOCK_SIDE, BLOCK_SIDE), inside_d, picture.width, picture.height);
  CHECK(cudaGetLastError());
  CHECK(cudaMemcpy(inside.data(), inside_d, threads * sizeof(int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(inside_d));
  return inside;
}

static Flags boundaryCheckOnCpu(const Picture& picture) {
  const int blocksX = blocksToCover(picture.width);
  const int blocksY = blocksToCover(picture.height);
  Flags inside;
  for (int by = 0; by < blocksY; ++by) {
    for (int bx = 0; bx < blocksX; ++bx) {
      for (int t = 0; t < THREADS_PER_BLOCK; ++t) {
        const int col = bx * BLOCK_SIDE + t % BLOCK_SIDE;
        const int row = by * BLOCK_SIDE + t / BLOCK_SIDE;
        inside.push_back(col < picture.width && row < picture.height ? INSIDE : OUTSIDE);
      }
    }
  }
  return inside;
}

static int lanesInside(const Flags& inside, int warp) {
  int lanes = 0;
  for (int i = warp * WARP_SIZE; i < (warp + 1) * WARP_SIZE; ++i) {
    lanes += inside[i];
  }
  return lanes;
}

struct WarpCounts {
  int fullyInside = 0;
  int divergent = 0;
  int fullyOutside = 0;
};

static WarpCounts countWarps(const Flags& inside, int firstWarp, int warps) {
  WarpCounts counts;
  for (int warp = firstWarp; warp < firstWarp + warps; ++warp) {
    const int lanes = lanesInside(inside, warp);
    if (lanes == WARP_SIZE) {
      ++counts.fullyInside;
    } else if (lanes == 0) {
      ++counts.fullyOutside;
    } else {
      ++counts.divergent;
    }
  }
  return counts;
}

static void printDivergentWarpsPerBlock(const Flags& inside, const Picture& picture) {
  const int blocksX = blocksToCover(picture.width);
  const int blocksY = blocksToCover(picture.height);
  std::printf("divergent warps of each block (of %d):\n", WARPS_PER_BLOCK);
  for (int by = 0; by < blocksY; ++by) {
    for (int bx = 0; bx < blocksX; ++bx) {
      const int block = by * blocksX + bx;
      std::printf(" %d", countWarps(inside, block * WARPS_PER_BLOCK, WARPS_PER_BLOCK).divergent);
    }
    std::printf("\n");
  }
}

static bool runPicture(const Picture& picture) {
  const int blocksX = blocksToCover(picture.width);
  const int blocksY = blocksToCover(picture.height);
  const int warps = blocksX * blocksY * WARPS_PER_BLOCK;

  const Flags inside = boundaryCheckOnDevice(picture);
  const Flags expected = boundaryCheckOnCpu(picture);
  const WarpCounts counts = countWarps(inside, 0, warps);

  std::printf("%d x %d picture: %d x %d blocks of %d x %d, %d warps\n",
              picture.width,
              picture.height,
              blocksX,
              blocksY,
              BLOCK_SIDE,
              BLOCK_SIDE,
              warps);
  printDivergentWarpsPerBlock(inside, picture);
  std::printf("fully inside %d, divergent %d (%.1f %%), fully outside %d\n\n",
              counts.fullyInside,
              counts.divergent,
              100.0 * counts.divergent / warps,
              counts.fullyOutside);
  return inside == expected;
}

int main() {
  std::printf("each warp is two rows of a %d x %d block; it diverges when some lanes are\n",
              BLOCK_SIDE,
              BLOCK_SIDE);
  std::printf("inside the picture and some outside\n\n");

  bool ok = true;
  for (const Picture& picture : PICTURES) {
    ok = runPicture(picture) && ok;
  }
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 08_divergence_2d.cu -o 08_divergence_2d && ./08_divergence_2d

**What to look at in the output:** Only the right-hand column of blocks diverges, 8 warps per block, because the edge cuts every pair of rows. The bottom row of blocks never does: each of its warps is fully inside or fully outside. 76 × 62 has 31 divergent warps of 160; 100 × 70 has 35 of 280 (12.5 %), the answer to exercise 12.8. Try: change the first picture to 76 × 63: row 62 now exists but row 63 does not, so the last warp of every bottom block splits and the count rises to 36.

## Step 9 of 9: `09_warp_efficiency.cu`

The kernel `k` of exercise 12.1 on N = 2048 with blocks of 128 threads. Every branch also marks the threads that take it, so the host can count, per warp, the active lanes at line 4, at line 6 and in each iteration of the loop on line 7. (slide: *Exercise 12.1: count warps, divergence and SIMD efficiency*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 09_warp_efficiency.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

const unsigned N = 2048;
const unsigned THREADS_PER_BLOCK = 128;
const unsigned BLOCKS = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
const int WARP_SIZE = 32;
const int WARPS_PER_BLOCK = THREADS_PER_BLOCK / WARP_SIZE;
const int WARPS = BLOCKS * WARPS_PER_BLOCK;
const int MAX_TRIPS = 4;
const int SHOWN = 8;

using Ints = std::vector<int>;

struct Paths {
  int* line4;
  int* line6;
  int* loopTrips;
};

__global__ void k(int* a, int* b, Paths paths) {
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (threadIdx.x < 48 || threadIdx.x >= 112) {
    b[i] = a[i] + 1;
    paths.line4[i] = 1;
  }
  if (i % 4 == 0) {
    a[i] = b[i] * 2;
    paths.line6[i] = 1;
  }
  for (unsigned int j = 0; j < 4 - (i % 4); ++j) {
    b[i] += j;
    paths.loopTrips[i] += 1;
  }
}

struct Run {
  Ints a = Ints(N);
  Ints b = Ints(N, 0);
  Ints line4 = Ints(N, 0);
  Ints line6 = Ints(N, 0);
  Ints loopTrips = Ints(N, 0);
};

static int* deviceCopy(const Ints& host) {
  int* device;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

static void copyBack(Ints& host, int* device) {
  CHECK(cudaMemcpy(host.data(), device, host.size() * sizeof(int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(device));
}

static void runOnDevice(Run& run) {
  int* a_d = deviceCopy(run.a);
  int* b_d = deviceCopy(run.b);
  const Paths paths_d{deviceCopy(run.line4), deviceCopy(run.line6), deviceCopy(run.loopTrips)};

  launch(k, dim3(BLOCKS), dim3(THREADS_PER_BLOCK), a_d, b_d, paths_d);
  CHECK(cudaGetLastError());

  copyBack(run.a, a_d);
  copyBack(run.b, b_d);
  copyBack(run.line4, paths_d.line4);
  copyBack(run.line6, paths_d.line6);
  copyBack(run.loopTrips, paths_d.loopTrips);
}

static void runOnCpu(Run& run) {
  for (unsigned i = 0; i < N; ++i) {
    const unsigned t = i % THREADS_PER_BLOCK;
    if (t < 48 || t >= 112) {
      run.b[i] = run.a[i] + 1;
      run.line4[i] = 1;
    }
    if (i % 4 == 0) {
      run.a[i] = run.b[i] * 2;
      run.line6[i] = 1;
    }
    for (unsigned j = 0; j < 4 - (i % 4); ++j) {
      run.b[i] += j;
      run.loopTrips[i] += 1;
    }
  }
}

static int lanesWhere(const Ints& values, int warp, int atLeast) {
  int lanes = 0;
  for (int i = warp * WARP_SIZE; i < (warp + 1) * WARP_SIZE; ++i) {
    lanes += values[i] >= atLeast ? 1 : 0;
  }
  return lanes;
}

static void printStatement(const char* name, const Ints& ran) {
  int active = 0;
  int divergent = 0;
  for (int warp = 0; warp < WARPS; ++warp) {
    const int lanes = lanesWhere(ran, warp, 1);
    active += lanes > 0 ? 1 : 0;
    divergent += lanes > 0 && lanes < WARP_SIZE ? 1 : 0;
  }
  std::printf(
      "%s: %d active warps, %d divergent; SIMD efficiency in block 0:", name, active, divergent);
  for (int warp = 0; warp < WARPS_PER_BLOCK; ++warp) {
    std::printf(" warp %d %3d %%", warp, 100 * lanesWhere(ran, warp, 1) / WARP_SIZE);
  }
  std::printf("\n");
}

static void printLoop(const Ints& loopTrips) {
  int uniform = 0;
  int diverging = 0;
  std::printf("(d) loop on line 7, lanes of warp 0 in each iteration:");
  for (int j = 0; j < MAX_TRIPS; ++j) {
    const int lanes = lanesWhere(loopTrips, 0, j + 1);
    std::printf(" j=%d: %d", j, lanes);
    uniform += lanes == WARP_SIZE ? 1 : 0;
    diverging += lanes > 0 && lanes < WARP_SIZE ? 1 : 0;
  }
  std::printf("\n    %d iteration without divergence, %d with\n", uniform, diverging);
}

static void printFirst(const char* label, const Ints& values) {
  std::printf("%-12s", label);
  for (int i = 0; i < SHOWN; ++i) {
    std::printf(" %3d", values[i]);
  }
  std::printf(" ...\n");
}

int main() {
  Run device;
  for (unsigned i = 0; i < N; ++i) {
    device.a[i] = static_cast<int>(i % 7);
  }
  Run cpu = device;
  printFirst("a before:", device.a);
  printFirst("b before:", device.b);

  runOnDevice(device);
  runOnCpu(cpu);
  printFirst("a after:", device.a);
  printFirst("b after:", device.b);

  std::printf("\n(a) N = %u, %u blocks of %u threads: %d warps per block, %d in the grid\n",
              N,
              BLOCKS,
              THREADS_PER_BLOCK,
              WARPS_PER_BLOCK,
              WARPS);
  printStatement("(b) line 4", device.line4);
  printStatement("(c) line 6", device.line6);
  printLoop(device.loopTrips);

  const bool ok = device.a == cpu.a && device.b == cpu.b && device.line4 == cpu.line4 &&
                  device.line6 == cpu.line6 && device.loopTrips == cpu.loopTrips;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 09_warp_efficiency.cu -o 09_warp_efficiency && ./09_warp_efficiency

**What to look at in the output:** The answers of exercise 12.1: (a) 4 warps per block and 64 in the grid; (b) 48 active and 32 divergent warps at line 4, with warp 2 of each block inactive but not divergent; (c) every warp runs line 6 with only 8 lanes, 25 %; (d) the loop runs 32, 24, 16 and 8 lanes of each warp, so 1 iteration without divergence and 3 with.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/02_gpu_architecture/README.md`.

**Next:** open `ec2/03_memory_tiling/colab_1_step_by_step.ipynb`.